In [8]:
from keras_preprocessing.image import ImageDataGenerator
import os
from tensorflow.keras.layers import Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.applications.resnet_v2 import ResNet152V2
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from matplotlib import pyplot
from PIL import ImageFile
import tensorflow as tf
import sys


In [9]:
train_dataset = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2)

In [10]:
train_it = train_dataset.flow_from_directory("C:/Gait/Train/", class_mode="categorical",
                                             batch_size=32, target_size=(128, 128), subset='training'
                                            )

Found 1419 images belonging to 3 classes.


In [11]:
test_it = train_dataset.flow_from_directory("C:/Gait/Train/", class_mode="categorical",
                                            batch_size=32, target_size=(128, 128), subset='validation'
                                            )


Found 353 images belonging to 3 classes.


In [19]:

# created optimizer for model compilation
optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)
# optimizer = tf.keras.optimizers.Adamax(learning_rate=0.01)



# # check if any previously trained model saved
# if os.path.isfile("./models/ImageClassifier.h5"):
#     model = tf.keras.models.load_model('./models/ImageClassifier.h5')

model = ResNet152V2(include_top=False, input_shape=(128, 128, 3))
# another option to use input shape as (200,200,3)

# freeze the all pretrained model layers
for layer in model.layers:
    layer.trainable = False

#     # this will un-freeze the pretrained model's normalized layers.
#     # pretty slow process but works fine with heavy number of classes
#     '''
#     if "BatchNormalization" in layer.__class__.__name__:
#         layer.trainable = True
#         '''

dropout = Dropout(0.1)(model.layers[-1].output)

# BatchNormalization for the output of the pretrained model
batch = BatchNormalization()(dropout)

# GlobalAveragePooling2D works better with ResNet152V2
pool = GlobalAveragePooling2D(name="avg_pool")(batch)
flat = Flatten()(pool)

dense = Dense(64, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)

dense = Dense(32)(dropout)
dropout = Dropout(0.05)(dense)

dense = Dense(24)(dropout)
dropout = Dropout(0.05)(dense)

# dense = Dense(36)(dropout)
# dropout = Dropout(0.05)(dense)

dense = Dense(18)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")



In [20]:
es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

history = model.fit(train_it,validation_data=test_it, validation_steps=len(test_it),callbacks=[es], epochs=25, verbose=1)




Epoch 1/25
45/45 [==============================] - 74s 1s/step - loss: 0.7878 - accuracy: 0.6413 - val_loss: 0.6419 - val_accuracy: 0.6941
Epoch 2/25
45/45 [==============================] - 64s 1s/step - loss: 0.5067 - accuracy: 0.7773 - val_loss: 0.6113 - val_accuracy: 0.7110
Epoch 3/25
45/45 [==============================] - 63s 1s/step - loss: 0.3742 - accuracy: 0.8520 - val_loss: 0.6950 - val_accuracy: 0.7110
Epoch 4/25
45/45 [==============================] - 62s 1s/step - loss: 0.2914 - accuracy: 0.8753 - val_loss: 1.0475 - val_accuracy: 0.6459
Epoch 4: early stopping


In [7]:
_, acc = model.evaluate(test_it, steps=len(test_it), verbose=2)
#     model.save("./Gait_Classifier.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))
# summarize_diagnostics(history)

12/12 - 27s - loss: 0.6436 - accuracy: 0.7592 - 27s/epoch - 2s/step
Finished training
> 75.921


NameError: name 'summarize_diagnostics' is not defined